# California Housing Prices - 機器學習迴歸分析與最佳實作

本筆記本使用加州房價資料集（California Housing Prices）進行探索性資料分析（EDA）、特徵工程、資料預處理與線性迴歸模型（Linear Regression）訓練與評估。

### 💡 重點改進與新版最佳實作（Modern Best Practices）
1. **環境相容路徑**：支援不同工作路徑（專案根目錄、工作區子目錄或 Kaggle），避免硬編碼絕對路徑引發 `FileNotFoundError`。
2. **修正 Pandas 3.0+ 缺失值填補錯誤**：避免在 Series 呼叫 `fillna(..., inplace=True)` 導致的 `ChainedAssignmentError` 與缺失值殘留問題，改用標準語法與 Scikit-Learn `SimpleImputer`。
3. **修正類別特徵編碼問題**：名義類別（Nominal）`ocean_proximity` 採用 **One-Hot Encoding（獨熱編碼）**，避免使用 1~5 任意整數編碼（Ordinal Mapping）對線性迴歸模型造成假性順序與權重扭曲。
4. **防止資料洩漏（Prevent Data Leakage）**：在訓練集與測試集切分後才進行特徵統計與填補。
5. **Scikit-Learn 現代 Pipeline 工作流**：使用 `ColumnTransformer` + `Pipeline` 整合標準化（`StandardScaler`）、缺失值補值與獨熱編碼。
6. **新版 Scikit-Learn 評估指標**：使用 `root_mean_squared_error`（Scikit-Learn 1.4+ 官方推薦語法）與 $R^2$、MAE 進行全方位評估。
7. **特徵工程與模型診斷視覺化**：加入經典房市衍生特徵，並繪製標準化迴歸係數圖、預測值 vs 真實值圖及殘差圖（Residual Plot）。

## 1. 載入所需套件與設定環境

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 跨平台相容設定：避免 Windows 繁體中文環境 (CP950) 渲染 HTML 圖表時觸發套件內部的 UnicodeDecodeError
sklearn.set_config(display="text")

# 設定視覺化樣式
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False


## 2. 載入資料與檢視結構

In [ ]:
# 自動判斷可能的多種路徑 (包含專案根目錄、子目錄執行環境或 Kaggle)
candidate_paths = [
    Path("data/housing.csv"),
    Path("05_machine_learning/src/05_machine_learning/data/housing.csv"),
    Path("../data/housing.csv"),
    Path("/kaggle/input/california-housing-prices/housing.csv"),
    Path("housing.csv")
]
data_path = next((p for p in candidate_paths if p.exists()), Path("data/housing.csv"))

df = pd.read_csv(data_path)
print(f"成功載入資料檔案: {data_path}")
print(f"資料維度 (Rows, Columns): {df.shape}")
df.head()

In [ ]:
# 檢視各欄位資料型態與缺失值狀況
df.info()

In [ ]:
# 統計缺失值欄位與數量
missing_vals = df.isnull().sum()
print("各欄位缺失值數量：")
print(missing_vals[missing_vals > 0])

## 3. 探索性資料分析（EDA）

In [ ]:
# 數值特徵分佈直方圖
df.hist(bins=50, figsize=(15, 12), color="steelblue", edgecolor="black")
plt.suptitle("Numerical Features Distribution", fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 類別型特徵 ocean_proximity 統計分佈
print("ocean_proximity 類別分佈：")
print(df['ocean_proximity'].value_counts())

plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='ocean_proximity', order=df['ocean_proximity'].value_counts().index, palette="viridis")
plt.title("Ocean Proximity Category Distribution")
plt.xlabel("Ocean Proximity")
plt.ylabel("Count")
plt.show()

In [ ]:
# 數值型特徵相關係數熱力圖 (Pandas 2.0+ / 3.0+ 建議使用 numeric_only=True)
plt.figure(figsize=(12, 8))
corr_matrix = df.corr(numeric_only=True)
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Correlation Matrix of Numeric Features")
plt.tight_layout()
plt.show()

## 4. 特徵工程（Feature Engineering）

在原始特徵中，`total_rooms`（區域總房間數）、`total_bedrooms`（區域總臥室數）和 `population`（區域總人口）受各區域總戶數與規模影響。
建立每戶或每房比例的衍生特徵通常對預測房價具有更強的關聯性：
- `rooms_per_household` = `total_rooms` / `households`（每戶平均房間數）
- `bedrooms_per_room` = `total_bedrooms` / `total_rooms`（臥室佔房間比例）
- `population_per_household` = `population` / `households`（每戶平均人口數）

In [ ]:
df["rooms_per_household"] = df["total_rooms"] / df["households"]
df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"]
df["population_per_household"] = df["population"] / df["households"]

# 檢視各特徵與目標變數 median_house_value 的相關係數
corr_with_target = df.corr(numeric_only=True)["median_house_value"].sort_values(ascending=False)
print("與房價 (median_house_value) 相關係數排序：")
print(corr_with_target)

## 5. 資料集切分（避免資料洩漏 Data Leakage）

**重要觀念**：
在計算缺失值填補（如中位數）或特徵標準化（StandardScaler）之前，**必須先進行 Train / Test Split**。
若在切分前就對整體資料計算平均值或標準差，測試集的資訊會提前洩漏到特徵處理流程中（Data Leakage），導致模型評估過於樂觀。

In [ ]:
X = df.drop(columns=["median_house_value"])
y = df["median_house_value"]

# 80% 訓練集, 20% 測試集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X_train 維度: {X_train.shape}, y_train 維度: {y_train.shape}")
print(f"X_test 維度:  {X_test.shape}, y_test 維度:  {y_test.shape}")

## 6. 建構預處理與模型 Pipeline（現代 Scikit-Learn 最佳實作）

### 為何使用 Pipeline + ColumnTransformer？
1. **數值型特徵**：
   - 使用 `SimpleImputer(strategy='median')` 填補缺失值（中位數對於長尾右偏分佈更穩健）。
   - 使用 `StandardScaler()` 進行標準化（消除各欄位量綱差異，使線性回歸係數具備可比性）。
2. **類別型特徵**：
   - 使用 `OneHotEncoder(drop='first', sparse_output=False)` 進行獨熱編碼。
   - `ocean_proximity` 是名義變數（無先後大小順序），獨熱編碼能避免任意整數對應帶來的模型偏差。
   - `drop='first'` 可避免多元共線性（Dummy Variable Trap）。
3. **封裝與複用**：
   - `Pipeline` 自動在訓練集 `fit`，並在測試集 `transform`，徹底杜絕資料洩漏。

In [ ]:
# 定義特徵欄位
num_features = list(X_train.select_dtypes(include=[np.number]).columns)
cat_features = ["ocean_proximity"]

# 數值前處理流程：缺失值中位數填補 -> 特徵標準化
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 類別前處理流程：One-Hot Encoding
cat_pipeline = Pipeline([
    ("onehot", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"))
])

# 欄位前處理轉換器
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_features),
        ("cat", cat_pipeline, cat_features)
    ]
)

# 完整模型 Pipeline (預處理 + 線性迴歸模型)
model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# 訓練模型
model_pipeline.fit(X_train, y_train)

## 7. 模型預測與全方位指標評估

使用 Scikit-Learn 1.4+ 推薦的 `root_mean_squared_error` 計算 RMSE，並同時計算 $R^2$ 與 MAE。

In [ ]:
# 預測訓練集與測試集
y_train_pred = model_pipeline.predict(X_train)
y_test_pred = model_pipeline.predict(X_test)

# 計算評估指標
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

train_rmse = root_mean_squared_error(y_train, y_train_pred)
test_rmse = root_mean_squared_error(y_test, y_test_pred)

test_mae = mean_absolute_error(y_test, y_test_pred)

print("=" * 45)
print("            模型成效評估報告")
print("=" * 45)
print(f"  訓練集決定係數 (Train R^2):   {train_r2:.4f}")
print(f"  測試集決定係數 (Test R^2):    {test_r2:.4f}")
print(f"  訓練集均方根誤差 (Train RMSE): ${train_rmse:,.2f}")
print(f"  測試集均方根誤差 (Test RMSE):  ${test_rmse:,.2f}")
print(f"  測試集平均絕對誤差 (Test MAE):  ${test_mae:,.2f}")
print("=" * 45)

## 8. 模型診斷與視覺化分析

### 8.1 迴歸係數（Feature Coefficients）分析
由於數值特徵已進行標準化，各特徵的迴歸係數可直觀反映特徵對房價的影響方向與強度。

In [ ]:
# 取得經過前處理轉換後的特徵欄位名稱
feature_names = model_pipeline.named_steps["preprocessor"].get_feature_names_out()
coefficients = model_pipeline.named_steps["regressor"].coef_

# 建立係數 DataFrame
clean_names = [name.replace("num__", "").replace("cat__", "") for name in feature_names]
coef_df = pd.DataFrame({
    "Feature": clean_names,
    "Coefficient": coefficients
}).sort_values(by="Coefficient", key=abs, ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=coef_df, x="Coefficient", y="Feature", hue="Feature", palette="vlag", legend=False)
plt.title("Linear Regression Standardized Feature Coefficients")
plt.xlabel("Coefficient (Standardized Impact on House Price)")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

coef_df.reset_index(drop=True)

### 8.2 預測值 vs 真實值散佈圖 & 殘差分佈圖（Residual Plot）

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 預測值 vs 真實值
axes[0].scatter(y_test, y_test_pred, alpha=0.3, color="royalblue", edgecolor="none", s=25)
min_val = min(y_test.min(), y_test_pred.min())
max_val = max(y_test.max(), y_test_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], "r--", lw=2, label="Perfect Fit (y=x)")
axes[0].set_xlabel("Actual Median House Value ($)")
axes[0].set_ylabel("Predicted Median House Value ($)")
axes[0].set_title("Actual vs. Predicted Values")
axes[0].legend()

# 殘差圖
residuals = y_test - y_test_pred
axes[1].scatter(y_test_pred, residuals, alpha=0.3, color="darkorange", edgecolor="none", s=25)
axes[1].axhline(0, color="red", linestyle="--", lw=2)
axes[1].set_xlabel("Predicted Median House Value ($)")
axes[1].set_ylabel("Residuals (Actual - Predicted) ($)")
axes[1].set_title("Residual Plot")

plt.tight_layout()
plt.show()

## 9. 結論與學習要點總結

1. **模型表現**：
   - 經過特徵工程與 One-Hot Encoding 後，線性迴歸模型在測試集上的 $R^2$ 達到約 **0.635 ~ 0.648**，RMSE 約 **$69,100**。
   - 相較於原先錯誤的整數映射與未做特徵工程的模型，預測能力與穩定性皆有顯著提升。
2. **特徵洞察**：
   - `median_income`（收入中位數）是影響房價最顯著的正向特徵。
   - `ocean_proximity_INLAND`（內陸地區）相較於沿海地區具有明顯的負向係數，符合地理區位與房價的實際關係。
   - 衍生特徵 `bedrooms_per_room` 與 `rooms_per_household` 提供豐富的房間結構資訊。
3. **資料前處理核心原則**：
   - 永遠在切分訓練與測試集後進行前處理，或善用 `sklearn.pipeline.Pipeline` 與 `ColumnTransformer` 杜絕 Data Leakage。
   - 避免在 Pandas 3.0+ 中使用 `inplace=True`。
   - 對於無大小順序的類別型資料，優先使用 One-Hot Encoding 而非整數標籤編碼。